# Notebook 15
## DNA DNABERT-2 Embedding Extraction

Extracts pretrained transformer embeddings using **DNABERT-2** (`zhihan1996/DNABERT-2-117M`).

### Known compatibility issues (transformers >= 4.40 / Triton 2.x)
1. `pad_token_id` AttributeError in `BertEmbeddings.__init__` -- fixed by Cell 0 patch.
2. ALiBi meta-device error in `rebuild_alibi_tensor` -- fixed by Cell 0 patch.
3. Triton `trans_b` CompilationError in `flash_attn_triton.py` -- fixed at runtime
   in Cell 5 by forcing PyTorch attention fallback (no file edits required).

**Workflow:** run Cell 0, restart kernel, then run Cells 1-12 in order.

### Outputs
- `data/processed/dna_dnabert2_embeddings_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `data/processed/dna_dnabert2_labels_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `data/processed/dna_dnabert2_ids_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `reports/dna_dnabert2_embeddings_summary.json`

## 0) One-time patch of cached `bert_layers.py`

**Run once, then restart the kernel.**

Re-downloads the original file from the Hub (clears any previous broken patches),
then applies two fixes:
- Fix 1: `config.pad_token_id` -> `getattr(config, 'pad_token_id', 0) or 0`
- Fix 2: `rebuild_alibi_tensor` rewritten with `device='cpu'` on all constructors
  and `.clone()` after `.expand()`.

The Triton `trans_b` issue is handled separately at runtime in Cell 5 --
no `flash_attn_triton.py` edits are needed.

In [1]:
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

# Re-download original to clear any previous broken patch attempts
print('Re-downloading original bert_layers.py...')
orig_path = hf_hub_download(
    repo_id='zhihan1996/DNABERT-2-117M',
    filename='bert_layers.py',
    force_download=True,
)
print('Downloaded to:', orig_path)

cache_root = Path.home() / '.cache' / 'huggingface' / 'modules' / 'transformers_modules'
hits = list(cache_root.glob('**/bert_layers.py'))
assert hits, 'No bert_layers.py in transformers_modules cache.'
target = hits[0]
shutil.copy2(orig_path, target)
print('Restored original. Target:', target)

text = target.read_text()

# Fix 1: pad_token_id
P1_OLD = 'padding_idx=config.pad_token_id'
P1_NEW = "padding_idx=(getattr(config, 'pad_token_id', 0) or 0)"
assert P1_OLD in text, 'Fix 1 target not found'
text = text.replace(P1_OLD, P1_NEW)
print('Fix 1 (pad_token_id): APPLIED')

# Fix 2: rebuild_alibi_tensor -- locate and replace entire function
lines = text.splitlines(keepends=True)
start_idx = None
for i, line in enumerate(lines):
    if 'def rebuild_alibi_tensor' in line:
        start_idx = i
        break
assert start_idx is not None, 'rebuild_alibi_tensor not found'

fn_indent = len(lines[start_idx]) - len(lines[start_idx].lstrip())
end_idx = len(lines)
for j in range(start_idx + 1, len(lines)):
    s = lines[j].strip()
    if not s or s.startswith('#'): continue
    li = len(lines[j]) - len(lines[j].lstrip())
    if li <= fn_indent and (s.startswith('def ') or s.startswith('class ')):
        end_idx = j
        break

ind = ' ' * fn_indent
i4  = ind + '    '
i8  = ind + '        '
i12 = ind + '            '

new_fn = ''.join([
    ind + 'def rebuild_alibi_tensor(\n',
    ind + '        self,\n',
    ind + '        size: int,\n',
    ind + '        device=None):  # _DNABERT2_PATCHED_v3\n',
    i4  + 'n_heads = self.num_attention_heads\n',
'\n',
    i4  + 'def _get_alibi_head_slopes(n_heads: int):\n',
'\n',
    i8  + 'def get_slopes_power_of_2(n_heads: int):\n',
    i12 + 'start = (2**(-2**-(math.log2(n_heads) - 3)))\n',
    i12 + 'ratio = start\n',
    i12 + 'return [start * ratio**i for i in range(n_heads)]\n',
'\n',
    i8  + 'if math.log2(n_heads).is_integer():\n',
    i12 + 'return get_slopes_power_of_2(n_heads)\n',
'\n',
    i8  + 'closest_power_of_2 = 2**math.floor(math.log2(n_heads))\n',
    i8  + 'slopes_a = get_slopes_power_of_2(closest_power_of_2)\n',
    i8  + 'slopes_b = _get_alibi_head_slopes(2 * closest_power_of_2)\n',
    i8  + 'slopes_b = slopes_b[0::2][:n_heads - closest_power_of_2]\n',
    i8  + 'return slopes_a + slopes_b\n',
'\n',
    i4  + 'ctx = torch.arange(size, dtype=torch.long, device="cpu")[:, None]\n',
    i4  + 'mem = torch.arange(size, dtype=torch.long, device="cpu")[None, :]\n',
    i4  + 'rel = torch.abs(mem - ctx).unsqueeze(0).expand(n_heads, -1, -1).clone()\n',
    i4  + 'slopes = torch.tensor(_get_alibi_head_slopes(n_heads), dtype=torch.float32, device="cpu")\n',
    i4  + 'alibi = slopes.unsqueeze(1).unsqueeze(1) * -rel.float()\n',
    i4  + 'alibi = alibi.unsqueeze(0)\n',
    i4  + 'assert alibi.shape == torch.Size([1, n_heads, size, size])\n',
    i4  + 'self._current_alibi_size = size\n',
    i4  + 'self.alibi = alibi\n',
])

new_lines = lines[:start_idx] + [new_fn + '\n'] + lines[end_idx:]
target.write_text(''.join(new_lines))
print(f'Fix 2 (ALiBi): APPLIED (replaced lines {start_idx+1}-{end_idx})')
print('\nBoth fixes written.')
print('>>> RESTART THE KERNEL, then run from Cell 1 onwards. <<<')


Re-downloading original bert_layers.py...


bert_layers.py: 0.00B [00:00, ?B/s]

Downloaded to: /home/dpratapa/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py
Restored original. Target: /home/dpratapa/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py
Fix 1 (pad_token_id): APPLIED
Fix 2 (ALiBi): APPLIED (replaced lines 362-406)

Both fixes written.
>>> RESTART THE KERNEL, then run from Cell 1 onwards. <<<


---
## After restarting the kernel, run from here
---

## 1) Imports

In [1]:
import json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


torch: 2.10.0+cu128
CUDA available: True


## 2) Paths, config, seed

In [2]:
ROOT      = Path.cwd().parents[0]
PROCESSED = ROOT / 'data' / 'processed'
REPORTS   = ROOT / 'reports'
CONFIGS   = ROOT / 'configs'
for p in [PROCESSED, REPORTS]: p.mkdir(parents=True, exist_ok=True)

with open(CONFIGS / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

SEED  = int(cfg['project']['random_seed'])
L     = int(cfg['dna']['seq_length_bp'])
N_POS = int(cfg['dna']['n_pos'])
N_NEG = int(cfg['dna']['n_neg'])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'SEED={SEED}  L={L}  N_POS={N_POS}  N_NEG={N_NEG}')


SEED=42  L=200  N_POS=2000  N_NEG=2000


## 3) Device

In [3]:
if torch.cuda.is_available():   DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print('Using device:', DEVICE)


Using device: cuda


## 4) Load dataset

In [4]:
in_csv = PROCESSED / f'dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv'
assert in_csv.exists(), f'Not found: {in_csv}'
df = pd.read_csv(in_csv)
assert (df['sequence'].str.len() != L).sum() == 0
print('Shape:', df.shape)
print(df['label'].value_counts().sort_index())


Shape: (4000, 6)
label
0    2000
1    2000
Name: count, dtype: int64


## 5) Load DNABERT-2 model

### Three compatibility issues handled here
1. `bert_layers.py` bugs are fixed by Cell 0 (kernel restart required).
2. **Triton `trans_b` CompilationError**: DNABERT-2 ships `flash_attn_triton.py` which
   calls `tl.dot(..., trans_b=True)`. This argument was removed in Triton 2.x.
   The model's own attention code already has a PyTorch fallback path:
   `if self.p_dropout or flash_attn_qkvpacked_func is None: # use PyTorch`.
   We activate this path by setting `p_dropout = 1e-9` on every attention layer.
   This value is only checked for branch selection; with `model.eval()` +
   `torch.no_grad()` dropout is never applied, so embeddings are unaffected.

In [5]:
MODEL_NAME = 'zhihan1996/DNABERT-2-117M'

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f'  pad_token_id: {tokenizer.pad_token_id}')

print(f'Loading model...')
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Disable Triton flash attention: set p_dropout > 0 to trigger PyTorch fallback.
# Has no effect on output quality -- dropout is inactive in eval + no_grad mode.
n_patched = 0
for module in model.modules():
    if type(module).__name__ == 'BertUnpadSelfAttention':
        module.p_dropout = 1e-9
        n_patched += 1
print(f'  Triton fallback: set p_dropout=1e-9 on {n_patched} attention layers -> PyTorch path')

model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Model dtype:      {next(model.parameters()).dtype}')


Loading tokenizer...
  pad_token_id: 3
Loading model...


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

BertModel LOAD REPORT from: zhihan1996/DNABERT-2-117M
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Triton fallback: set p_dropout=1e-9 on 12 attention layers -> PyTorch path
Model parameters: 117,068,544
Model dtype:      torch.float32


## 6) Tokenization check

In [6]:
enc = tokenizer(df['sequence'].iloc[0].upper(), return_tensors='pt',
                padding=False, truncation=True, max_length=512)
tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].tolist())
print('Sequence length (bp):  ', len(df['sequence'].iloc[0]))
print('input_ids shape:       ', enc['input_ids'].shape)
print('attention_mask sum:    ', enc['attention_mask'].sum().item())
print('First 20 BPE tokens:   ', tokens[:20])


Sequence length (bp):   200
input_ids shape:        torch.Size([1, 46])
attention_mask sum:     46
First 20 BPE tokens:    ['[CLS]', 'TGCC', 'CGC', 'GGACCTT', 'GCC', 'GCCCC', 'GCCTCCA', 'GCC', 'CGTG', 'CCACGG', 'CGGCC', 'GCCATT', 'GGCGC', 'GGGCC', 'CATCC', 'CAGAA', 'CGG', 'CGCC', 'CATT', 'GGCC']


## 7) Embedding extraction

DNABERT-2 `BertModel.forward` returns `(sequence_output, pooled_output)` -- a plain
tuple. `outputs[0]` is the last hidden state, shape `(batch, seq_len, hidden_dim)`.
We apply mask-weighted mean pooling over the token dimension.

In [7]:
def mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def extract_dnabert2_embeddings(sequences, tokenizer, model, device,
                                batch_size=32, max_length=512):
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size), desc='Extracting'):
            batch = [s.upper() for s in sequences[i:i+batch_size]]
            enc = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=max_length)
            ids  = enc['input_ids'].to(device)
            mask = enc['attention_mask'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            pooled = mean_pool(out[0], mask)  # out[0] = sequence_output
            all_embeddings.append(pooled.cpu().float().numpy())
    return np.concatenate(all_embeddings, axis=0)


## 8) Run extraction

A100 (40 GB) with batch_size=64: ~30 s for 4000 sequences.
Reduce BATCH_SIZE if you see CUDA OOM.

In [8]:
BATCH_SIZE = 32  # 16 GB -> 32; A100 -> 64
sequences = df['sequence'].tolist()

embeddings = extract_dnabert2_embeddings(
    sequences, tokenizer, model, DEVICE, BATCH_SIZE, 512)

print('Embeddings shape:', embeddings.shape)
print('Dtype:           ', embeddings.dtype)


Extracting:   0%|          | 0/125 [00:00<?, ?it/s]

Embeddings shape: (4000, 768)
Dtype:            float32


## 9) Sanity checks

In [9]:
assert embeddings.shape[0] == len(df)
assert not np.isnan(embeddings).any(), 'NaN!'
assert not np.isinf(embeddings).any(), 'Inf!'
print('Shape:', embeddings.shape, ' dtype:', embeddings.dtype)
print(f'Mean {embeddings.mean():.4f}  std {embeddings.std():.4f}  '
      f'min {embeddings.min():.4f}  max {embeddings.max():.4f}')
print('NaN:', np.isnan(embeddings).any(), ' Inf:', np.isinf(embeddings).any())


Shape: (4000, 768)  dtype: float32
Mean 0.0037  std 0.0974  min -5.6617  max 3.3474
NaN: False  Inf: False


## 10) Save arrays

In [10]:
labels = df['label'].astype(int).values
ids    = df['region_id'].astype(str).values
sfx = f'len{L}_pos{N_POS}_neg{N_NEG}'
emb_path    = PROCESSED / f'dna_dnabert2_embeddings_{sfx}.npy'
labels_path = PROCESSED / f'dna_dnabert2_labels_{sfx}.npy'
ids_path    = PROCESSED / f'dna_dnabert2_ids_{sfx}.npy'
np.save(emb_path,    embeddings.astype(np.float32))
np.save(labels_path, labels)
np.save(ids_path,    ids)
print('Saved:', emb_path)
print('Saved:', labels_path)
print('Saved:', ids_path)


Saved: /home/dpratapa/Capstone/data/processed/dna_dnabert2_embeddings_len200_pos2000_neg2000.npy
Saved: /home/dpratapa/Capstone/data/processed/dna_dnabert2_labels_len200_pos2000_neg2000.npy
Saved: /home/dpratapa/Capstone/data/processed/dna_dnabert2_ids_len200_pos2000_neg2000.npy


## 11) Reload verification

In [11]:
Xc = np.load(emb_path); yc = np.load(labels_path)
ic = np.load(ids_path, allow_pickle=True)
assert Xc.shape == embeddings.shape
assert (yc == labels).all()
assert not np.isnan(Xc).any()
print('Reload OK. Embeddings:', Xc.shape, ' Labels:', yc.shape)


Reload OK. Embeddings: (4000, 768)  Labels: (4000,)


## 12) Summary JSON

In [12]:
lc = {int(k): int(v) for k,v in zip(*np.unique(labels, return_counts=True))}
summary = {
    'notebook': '15_dna_dnabert2_embeddings',
    'model': MODEL_NAME,
    'patches': [
        'bert_layers.py: pad_token_id getattr fallback',
        'bert_layers.py: rebuild_alibi_tensor device=cpu + clone',
        'runtime: p_dropout=1e-9 to disable Triton flash_attn (trans_b removed in Triton 2.x)',
    ],
    'pooling': 'mean over outputs[0] (mask-weighted)',
    'embedding_dim': int(embeddings.shape[1]),
    'n_sequences': int(embeddings.shape[0]),
    'label_counts': lc, 'seq_length_bp': L,
    'batch_size': BATCH_SIZE, 'device': DEVICE, 'seed': SEED,
    'stats': {'mean': float(embeddings.mean()), 'std': float(embeddings.std()),
              'min': float(embeddings.min()), 'max': float(embeddings.max())},
    'files': {'embeddings': str(emb_path), 'labels': str(labels_path), 'ids': str(ids_path)},
    'timestamp': pd.Timestamp.now().isoformat(),
}
sp = REPORTS / 'dna_dnabert2_embeddings_summary.json'
sp.write_text(json.dumps(summary, indent=2))
print('Saved:', sp)
print(json.dumps(summary, indent=2))


Saved: /home/dpratapa/Capstone/reports/dna_dnabert2_embeddings_summary.json
{
  "notebook": "15_dna_dnabert2_embeddings",
  "model": "zhihan1996/DNABERT-2-117M",
  "patches": [
    "bert_layers.py: pad_token_id getattr fallback",
    "bert_layers.py: rebuild_alibi_tensor device=cpu + clone",
    "runtime: p_dropout=1e-9 to disable Triton flash_attn (trans_b removed in Triton 2.x)"
  ],
  "pooling": "mean over outputs[0] (mask-weighted)",
  "embedding_dim": 768,
  "n_sequences": 4000,
  "label_counts": {
    "0": 2000,
    "1": 2000
  },
  "seq_length_bp": 200,
  "batch_size": 32,
  "device": "cuda",
  "seed": 42,
  "stats": {
    "mean": 0.003702780930325389,
    "std": 0.09741873294115067,
    "min": -5.661664962768555,
    "max": 3.347414970397949
  },
  "files": {
    "embeddings": "/home/dpratapa/Capstone/data/processed/dna_dnabert2_embeddings_len200_pos2000_neg2000.npy",
    "labels": "/home/dpratapa/Capstone/data/processed/dna_dnabert2_labels_len200_pos2000_neg2000.npy",
    "ids

## Next

Proceed to `16_dna_dnabert2_models.ipynb`:
- Loads embeddings + labels
- `train_test_split(random_state=SEED, test_size=0.2, stratify=y)`
- Trains: Logistic Regression, SVM, Random Forest, XGBoost
- Reports: accuracy, precision, recall, F1, ROC-AUC
- Saves: CSV + JSON